# Experimentos GraphRAG — Grid de Configurações
**Objetivo:** Avaliar 8 sistemas (4 modos × 2 community-levels) nos 141 prontuários do ground truth.

| Sistema | Modo | Community Level |
|---|---|---|
| S1 | basic | 1 |
| S2 | basic | 2 |
| S3 | local | 1 |
| S4 | local | 2 |
| S5 | global | 1 |
| S6 | global | 2 |
| S7 | drift | 1 (default) |
| S8 | drift | 2 |

**Progresso salvo automaticamente** — pode interromper e continuar a qualquer momento.

## Célula 1 — Configuração

In [ ]:
import subprocess
import json
import time
import os
import re
import pandas as pd
from datetime import datetime
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets

# ── AJUSTE ESTES CAMINHOS ────────────────────────────────────────────────────
GRAPHRAG_ROOT = r'/mnt/c/Users/Matheus/Documents/trabalho de conclusão de curso/anamnese/experimentos/local copy'
GRAPHRAG_DATA = r'/mnt/c/Users/Matheus/Documents/trabalho de conclusão de curso/anamnese/experimentos/local copy/output'
GROUND_TRUTH  = './progresso/ground_truth_anotado.csv'
CSV_ORIGINAL  = './experimentos/local/input/data-hepato-sem-anonimizacao.csv'
COL_ID        = 'prontuario'
COL_TEXTO     = 'descricao'

# Pasta onde os resultados serão salvos
OUTPUT_DIR    = './resultados_grid'
PROGRESSO_DIR = './progresso_grid'

# Timeout por query (segundos) — DRIFT pode precisar de mais
TIMEOUT_BASIC_LOCAL = 120
TIMEOUT_GLOBAL      = 300
TIMEOUT_DRIFT       = 600

# Pausa entre queries (segundos) — evita rate limiting
PAUSA_ENTRE_QUERIES = 3
# ────────────────────────────────────────────────────────────────────────────

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(PROGRESSO_DIR).mkdir(parents=True, exist_ok=True)

# Grid de experimentos
SISTEMAS = [
    {'id': 'S1', 'method': 'basic',  'community_level': 1},
    {'id': 'S2', 'method': 'basic',  'community_level': 2},
    {'id': 'S3', 'method': 'local',  'community_level': 1},
    {'id': 'S4', 'method': 'local',  'community_level': 2},
    {'id': 'S5', 'method': 'global', 'community_level': 1},
    {'id': 'S6', 'method': 'global', 'community_level': 2},
    {'id': 'S7', 'method': 'drift',  'community_level': 1},
    {'id': 'S8', 'method': 'drift',  'community_level': 2},
]

DOENCAS_VALIDAS = [
    'Cirrose hepática',
    'Esquistossomose',
    'Hepatite B',
    'Hepatite C',
    'Esteatose hepática',
    'Hipertensão portal',
    'Hemorragia digestiva alta',
    'Peritonite bacteriana espontânea',
    'Nenhuma',
]

print('✔ Configuração carregada.')
print(f'  Output dir    : {OUTPUT_DIR}')
print(f'  Progresso dir : {PROGRESSO_DIR}')
print(f'  Sistemas      : {len(SISTEMAS)}')

: 

## Célula 2 — Carrega dados

In [ ]:
# Ground truth
gt = pd.read_csv(GROUND_TRUTH, dtype=str).fillna('')
gt['prontuario_id'] = gt['prontuario_id'].str.strip()
lista_prontuarios = gt['prontuario_id'].tolist()

# Textos originais
df_orig = pd.read_csv(CSV_ORIGINAL, dtype=str).fillna('')
df_orig[COL_ID] = df_orig[COL_ID].str.strip()

# Agrega textos por prontuário (pode ter múltiplas consultas)
texto_por_id = (
    df_orig.groupby(COL_ID)[COL_TEXTO]
    .apply(lambda x: '\n--- Atendimento ---\n'.join(x.tolist()))
    .to_dict()
)

print(f'Ground truth   : {len(gt)} prontuários')
print(f'Textos carregados : {len(texto_por_id)}')

# Verifica prontuários sem texto
sem_texto = [p for p in lista_prontuarios if p not in texto_por_id]
if sem_texto:
    print(f'⚠  {len(sem_texto)} prontuários sem texto: {sem_texto[:5]}')
else:
    print('✔ Todos os prontuários têm texto disponível.')

## Célula 3 — Funções principais

In [ ]:
def monta_prompt(pront_id: str, texto: str) -> str:
    """Monta o prompt clínico para classificação."""
    doencas = ', '.join(DOENCAS_VALIDAS[:-1]) + ' ou Nenhuma'
    return (
        f'Abaixo estão as evoluções/anamneses de um único paciente, '
        f'extraídas do prontuário eletrônico (nº {pront_id}). '
        f'Com base EXCLUSIVAMENTE nesse conteúdo (usando seu conhecimento médico geral apenas para interpretá-lo), '
        f'qual é a doença hepática principal desse paciente? '
        f'Escolha apenas uma entre: {doencas}. '
        f'Se nenhuma dessas condições estiver claramente documentada, responda \'Nenhuma\'. '
        f'Responda em uma frase curta, citando apenas o nome da doença.\n'
        f'--- PRONTUÁRIO {pront_id} ---\n'
        f'{texto[:8000]}\n'   # limita para não estourar contexto
        f'--- FIM DO PRONTUÁRIO {pront_id} ---'
    )


def extrai_doenca(resposta: str) -> str:
    """Extrai a doença da resposta do GraphRAG."""
    if not resposta:
        return 'ERRO'
    resp_lower = resposta.lower()
    MAPA = {
        'CIRROSE': 'cirrose_hepatica', 'CIRROSE HEPATICA': 'cirrose_hepatica',
        'CIRROSE HEPÁTICA': 'cirrose_hepatica', 'CIRROSE HEPÁTICA DESCOMPENSADA': 'cirrose_hepatica',
        'CIRROSE HEPÁTICA ALCOÓLICA': 'cirrose_hepatica', 'CIRROSE HEPÁTICA VIRAL C': 'cirrose_hepatica',
        'CIRROSE MICRONODULAR': 'cirrose_hepatica', 'CIRROSE DESCOMPENSADA': 'cirrose_hepatica',
        'CIRROSE HEPÁTICA CRÔNICA': 'cirrose_hepatica',
        'HIPERTENSAO PORTAL': 'hipertensao_portal', 'HIPERTENSÃO PORTAL': 'hipertensao_portal',
        'HIPERTENSÃO PORTA': 'hipertensao_portal', 'SINDROME DE HIPERTENSAO PORTAL': 'hipertensao_portal',
        'SÍNDROME DE HIPERTENSÃO PORTAL': 'hipertensao_portal', 'SÍNDROME DA HIPERTENSÃO PORTAL': 'hipertensao_portal',
        'HIPERTENSÃO PORTAL NÃO CIRRÓTICA': 'hipertensao_portal', 'HIPERTENSÃO PORTAL NÃO-CIRRÓTICA': 'hipertensao_portal',
        'EHE/HP': 'hipertensao_portal',
        'ESQUISTOSSOMOSE': 'esquistossomose', 'EQUISTOSSOMOSE': 'esquistossomose',
        'ESQUISTOSSOMOSE HEPATOESPLENICA': 'esquistossomose', 'ESQUISTOSSOMOSE HEPATOESPLÊNICA': 'esquistossomose',
        'ESQUISTOSSOMOSE HEPATOESPLÊNICA (EHE)': 'esquistossomose', 'ESQUISTOSSOMOSE MANSONI': 'esquistossomose',
        'ESQUISTOSSOMOSE MANSÔNICA': 'esquistossomose', 'EHE': 'esquistossomose',
        'HEPATITE C': 'hepatite_c', 'HEPATITE C (HCV)': 'hepatite_c',
        'HEPATITE C CRÔNICA': 'hepatite_c', 'HEPATITE CRÔNICA VIRAL': 'hepatite_c',
        'HEPATITE B': 'hepatite_b', 'HEPATITE B (HBV)': 'hepatite_b',
        'HEPATITE B CRONICA': 'hepatite_b', 'HEPATITE B INATIVA': 'hepatite_b',
        'HEPATITE B RESOLVIDA': 'hepatite_b', 'HEPATITE CRÔNICA B': 'hepatite_b',
        'HEPATITE CRÔNICA VIRAL B': 'hepatite_b', 'VÍRUS B (HEPATITE B)': 'hepatite_b',
        'ESTEATOSE': 'esteatose_hepatica', 'ESTEATOSE HEPATICA': 'esteatose_hepatica',
        'ESTEATOSE HEPÁTICA': 'esteatose_hepatica', 'ESTEATOSE HEPATICA DIFUSA': 'esteatose_hepatica',
        'DHGNA': 'esteatose_hepatica', 'NAFLD': 'esteatose_hepatica',
        'MASLD': 'esteatose_hepatica', 'MASH': 'esteatose_hepatica', 'NASH': 'esteatose_hepatica',
        'DOENÇA HEPÁTICA GORDUROSA NÃO ALCOÓLICA': 'esteatose_hepatica',
        'DOENÇA HEPÁTICA GORDUROSA NÃO ALCOÓLICA (DHGNA)': 'esteatose_hepatica',
        'DOENÇA HEPÁTICA ESTEATÓTICA METABÓLICA': 'esteatose_hepatica',
        'DOENÇA HEPÁTICA ESTEATÓTICA METABÓLICA (MASH)': 'esteatose_hepatica',
        'ESTEATO-HEPATITE NÃO ALCOÓLICA': 'esteatose_hepatica',
        'ESTEATO-HEPATITE NÃO ALCOÓLICA (NASH)': 'esteatose_hepatica',
        'ESTEATOHEPATITE': 'esteatose_hepatica', 'ESTEATOHEPATITE NÃO ALCOÓLICA': 'esteatose_hepatica',
        'HDA': 'hemorragia_digestiva_alta', 'HEMORRAGIA DIGESTIVA ALTA': 'hemorragia_digestiva_alta',
        'HDA (HEMORRAGIA DIGESTIVA ALTA)': 'hemorragia_digestiva_alta', 'HDA VARICOSA': 'hemorragia_digestiva_alta',
        'HEMORRAGIA DIGESTIVA ALTA VARICOSA': 'hemorragia_digestiva_alta', 'HEMORRAGIA VARICOSA': 'hemorragia_digestiva_alta',
        'PBE': 'pbe', 'PERITONITE BACTERIANA ESPONTÂNEA': 'pbe',
        'PBE (PERITONITE BACTERIANA ESPONTÂNEA)': 'pbe', 'PERITONITE BACTERIANA ESPONTANEA': 'pbe',
    }
    for termo, classe in mapa.items():
        if termo in resp_lower:
            return classe
    return 'nao_identificado'


def get_timeout(method: str) -> int:
    if method == 'drift':
        return TIMEOUT_DRIFT
    elif method == 'global':
        return TIMEOUT_GLOBAL
    return TIMEOUT_BASIC_LOCAL


def roda_query(pront_id: str, texto: str, method: str, community_level: int) -> dict:
    """Executa uma query no GraphRAG e retorna o resultado."""
    prompt = monta_prompt(pront_id, texto)
    cmd = [
        'uv', 'run', '--no-sync', 'graphrag', 'query',
        '--root', GRAPHRAG_ROOT,
        '--data', GRAPHRAG_DATA,
        '--method', method,
        '--community-level', str(community_level),
        '--response-type', 'Single sentence',
        '--query', prompt,
    ]

    inicio = time.time()
    try:
        resultado = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=get_timeout(method),
            encoding='utf-8',
            errors='replace',
        )
        duracao = round(time.time() - inicio, 1)

        # Extrai a resposta do stdout
        stdout = resultado.stdout or ''
        # GraphRAG imprime a resposta após "SUCCESS:"
        match = re.search(r'SUCCESS:\s*(.+)', stdout, re.IGNORECASE | re.DOTALL)
        resposta_raw = match.group(1).strip() if match else stdout.strip()[-500:]

        doenca = extrai_doenca(resposta_raw)
        status = 'ok' if resultado.returncode == 0 else 'erro_returncode'

        return {
            'prontuario_id': pront_id,
            'doenca_prevista': doenca,
            'resposta_raw': resposta_raw[:500],
            'duracao_s': duracao,
            'status': status,
            'returncode': resultado.returncode,
            'timestamp': datetime.now().isoformat(),
        }

    except subprocess.TimeoutExpired:
        return {
            'prontuario_id': pront_id,
            'doenca_prevista': 'TIMEOUT',
            'resposta_raw': '',
            'duracao_s': get_timeout(method),
            'status': 'timeout',
            'returncode': -1,
            'timestamp': datetime.now().isoformat(),
        }
    except Exception as e:
        return {
            'prontuario_id': pront_id,
            'doenca_prevista': 'ERRO',
            'resposta_raw': str(e)[:200],
            'duracao_s': round(time.time() - inicio, 1),
            'status': f'excecao: {type(e).__name__}',
            'returncode': -1,
            'timestamp': datetime.now().isoformat(),
        }


def arquivo_progresso(sistema_id: str) -> str:
    return os.path.join(PROGRESSO_DIR, f'progresso_{sistema_id}.json')


def carrega_progresso(sistema_id: str) -> dict:
    path = arquivo_progresso(sistema_id)
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}


def salva_progresso(sistema_id: str, progresso: dict):
    with open(arquivo_progresso(sistema_id), 'w', encoding='utf-8') as f:
        json.dump(progresso, f, ensure_ascii=False, indent=2)


def salva_csv_sistema(sistema_id: str, progresso: dict, sistema_cfg: dict):
    """Salva CSV parcial/final de um sistema."""
    linhas = list(progresso.values())
    if not linhas:
        return
    df = pd.DataFrame(linhas)
    df['method']          = sistema_cfg['method']
    df['community_level'] = sistema_cfg['community_level']
    df['sistema_id']      = sistema_id
    path = os.path.join(OUTPUT_DIR, f'resultado_{sistema_id}.csv')
    df.to_csv(path, index=False, encoding='utf-8-sig')


print('✔ Funções carregadas.')

## Célula 4 — Status atual do progresso

In [ ]:
print('Status atual dos experimentos:\n')
print(f'{"Sistema":<8} {"Modo":<10} {"Level":<8} {"Concluídos":<12} {"Pendentes":<10} {"Timeouts":<10} {"Erros"}')
print('-' * 65)

for s in SISTEMAS:
    prog = carrega_progresso(s['id'])
    concluidos = len(prog)
    pendentes  = len(lista_prontuarios) - concluidos
    timeouts   = sum(1 for v in prog.values() if v.get('status') == 'timeout')
    erros      = sum(1 for v in prog.values() if 'erro' in v.get('status', '') or 'excecao' in v.get('status', ''))
    print(f"{s['id']:<8} {s['method']:<10} {s['community_level']:<8} {concluidos:<12} {pendentes:<10} {timeouts:<10} {erros}")

print(f'\nTotal de prontuários por sistema: {len(lista_prontuarios)}')
print(f'Total de execuções no grid      : {len(lista_prontuarios) * len(SISTEMAS)}')

## Célula 5 — Execução do grid

**Configure `SISTEMAS_RODAR` para escolher quais sistemas executar.**
Use `'todos'` para rodar o grid completo, ou liste os IDs: `['S1', 'S2']`

In [ ]:
# ── CONFIGURE AQUI ───────────────────────────────────────────────
SISTEMAS_RODAR = 'todos'   # 'todos' ou lista: ['S1', 'S3', 'S7']
MODO_TESTE     = False     # True = roda só 3 prontuários por sistema (piloto)
# ────────────────────────────────────────────────────────────────

sistemas_selecionados = SISTEMAS if SISTEMAS_RODAR == 'todos' else [
    s for s in SISTEMAS if s['id'] in SISTEMAS_RODAR
]

prontuarios_rodar = lista_prontuarios[:3] if MODO_TESTE else lista_prontuarios

print(f'Sistemas selecionados : {[s["id"] for s in sistemas_selecionados]}')
print(f'Prontuários por sistema: {len(prontuarios_rodar)}')
print(f'Total de execuções    : {len(sistemas_selecionados) * len(prontuarios_rodar)}')

# Estimativa de tempo
tempos_estimados = {'basic': 200, 'local': 300, 'global': 300, 'drift': 2400}
tempo_total = sum(
    tempos_estimados[s['method']] * len(prontuarios_rodar)
    for s in sistemas_selecionados
)
print(f'Tempo estimado        : {tempo_total//3600}h {(tempo_total%3600)//60}min')

confirma = input('\nConfirmar execução? [s/N]: ').strip().lower()
if confirma != 's':
    print('Execução cancelada.')
else:
    inicio_total = time.time()

    for idx_s, sistema in enumerate(sistemas_selecionados):
        sid     = sistema['id']
        method  = sistema['method']
        level   = sistema['community_level']

        print(f'\n{"="*60}')
        print(f'Sistema {sid} — {method.upper()} / community-level {level}')
        print(f'{"="*60}')

        progresso = carrega_progresso(sid)
        pendentes = [p for p in prontuarios_rodar if p not in progresso]

        if not pendentes:
            print(f'  ✔ Já concluído ({len(progresso)} prontuários). Pulando.')
            continue

        print(f'  Já concluídos : {len(progresso)}')
        print(f'  Pendentes     : {len(pendentes)}')

        for idx_p, pront_id in enumerate(pendentes):
            texto = texto_por_id.get(pront_id, '')
            if not texto:
                progresso[pront_id] = {
                    'prontuario_id': pront_id,
                    'doenca_prevista': 'SEM_TEXTO',
                    'resposta_raw': '',
                    'duracao_s': 0,
                    'status': 'sem_texto',
                    'returncode': -1,
                    'timestamp': datetime.now().isoformat(),
                }
                salva_progresso(sid, progresso)
                continue

            # Log de progresso
            pct = (idx_p + 1) / len(pendentes) * 100
            elapsed = time.time() - inicio_total
            print(f'  [{idx_p+1:>3}/{len(pendentes)}] {pct:>5.1f}% | '
                  f'ID: {pront_id} | '
                  f'Elapsed: {int(elapsed//60)}m{int(elapsed%60):02d}s',
                  end=' ... ', flush=True)

            # Executa a query
            resultado = roda_query(pront_id, texto, method, level)

            # Log do resultado
            status_icon = '✔' if resultado['status'] == 'ok' else '⚠' if resultado['status'] == 'timeout' else '✗'
            print(f"{status_icon} {resultado['doenca_prevista']:<30} [{resultado['duracao_s']}s]")

            # Salva progresso
            progresso[pront_id] = resultado
            salva_progresso(sid, progresso)
            salva_csv_sistema(sid, progresso, sistema)

            # Pausa entre queries
            if idx_p < len(pendentes) - 1:
                time.sleep(PAUSA_ENTRE_QUERIES)

        # Resumo do sistema
        total    = len(progresso)
        ok       = sum(1 for v in progresso.values() if v['status'] == 'ok')
        timeouts = sum(1 for v in progresso.values() if v['status'] == 'timeout')
        erros    = total - ok - timeouts
        print(f'\n  Sistema {sid} concluído: {ok} OK | {timeouts} timeouts | {erros} erros')
        salva_csv_sistema(sid, progresso, sistema)

    tempo_total_real = time.time() - inicio_total
    print(f'\n{"="*60}')
    print(f'Grid concluído em {int(tempo_total_real//3600)}h {int((tempo_total_real%3600)//60)}min')
    print(f'{"="*60}')

## Célula 6 — Consolida todos os resultados

In [ ]:
dfs = []
for sistema in SISTEMAS:
    path = os.path.join(OUTPUT_DIR, f'resultado_{sistema["id"]}.csv')
    if os.path.exists(path):
        df_s = pd.read_csv(path, dtype=str).fillna('')
        dfs.append(df_s)
        print(f'  {sistema["id"]} ({sistema["method"]} / level {sistema["community_level"]}): {len(df_s)} linhas')
    else:
        print(f'  {sistema["id"]}: ainda não executado')

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    df_all.to_csv(os.path.join(OUTPUT_DIR, 'resultados_consolidados.csv'),
                  index=False, encoding='utf-8-sig')
    print(f'\n✔ Consolidado salvo: {len(df_all)} linhas totais')

    # Pivot: prontuário × sistema → doença prevista
    df_pivot = df_all.pivot_table(
        index='prontuario_id',
        columns='sistema_id',
        values='doenca_prevista',
        aggfunc='first',
    ).reset_index()

    # Adiciona ground truth
    gt_merge = gt[['prontuario_id', 'doenca_principal']]
    df_pivot = df_pivot.merge(gt_merge, on='prontuario_id', how='left')
    df_pivot.to_csv(os.path.join(OUTPUT_DIR, 'pivot_sistemas.csv'),
                    index=False, encoding='utf-8-sig')
    print(f'Pivot salvo: {df_pivot.shape}')
    display(df_pivot.head(5))
else:
    print('Nenhum resultado disponível ainda.')

## Célula 7 — Métricas por sistema

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, matthews_corrcoef
import numpy as np

def bootstrap_f1(y_true, y_pred, n=5000):
    scores = []
    yt, yp = np.array(y_true), np.array(y_pred)
    for _ in range(n):
        idx = np.random.choice(len(yt), len(yt), replace=True)
        try:
            scores.append(f1_score(yt[idx], yp[idx], average='macro', zero_division=0))
        except: pass
    return np.percentile(scores, [2.5, 97.5])

resultados_metricas = []

if 'df_pivot' in dir() and len(df_pivot) > 0:
    df_eval = df_pivot[df_pivot['doenca_principal'] != 'nenhuma'].copy()
    y_true  = df_eval['doenca_principal'].tolist()

    colunas_sistemas = [s['id'] for s in SISTEMAS if s['id'] in df_pivot.columns]

    for sid in colunas_sistemas:
        cfg = next(s for s in SISTEMAS if s['id'] == sid)
        y_pred = df_eval[sid].fillna('nao_identificado').tolist()

        acc   = accuracy_score(y_true, y_pred)
        f1m   = f1_score(y_true, y_pred, average='macro',    zero_division=0)
        f1w   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(y_true, y_pred)
        mcc   = matthews_corrcoef(y_true, y_pred)
        ic    = bootstrap_f1(y_true, y_pred)

        resultados_metricas.append({
            'Sistema':         sid,
            'Modo':            cfg['method'],
            'Community Level': cfg['community_level'],
            'Accuracy':        round(acc, 4),
            'F1 macro':        round(f1m, 4),
            'F1 weighted':     round(f1w, 4),
            "Cohen's Kappa":   round(kappa, 4),
            'MCC':             round(mcc, 4),
            'F1 IC 2.5%':      round(ic[0], 4),
            'F1 IC 97.5%':     round(ic[1], 4),
        })

    df_metricas = pd.DataFrame(resultados_metricas).sort_values('F1 macro', ascending=False)
    df_metricas.to_csv(os.path.join(OUTPUT_DIR, 'metricas_grid.csv'), index=False, encoding='utf-8-sig')

    print('Métricas por sistema (ordenado por F1 macro):\n')
    display(df_metricas)
else:
    print('Execute a célula 6 primeiro.')

## Célula 8 — Testes estatísticos (Friedman + Wilcoxon)

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

if 'df_pivot' in dir() and len(colunas_sistemas) >= 2:
    df_eval = df_pivot[df_pivot['doenca_principal'] != 'nenhuma'].copy()
    y_true  = df_eval['doenca_principal'].tolist()

    # Acerto por prontuário por sistema (1 = acertou, 0 = errou)
    acertos = {}
    for sid in colunas_sistemas:
        y_pred = df_eval[sid].fillna('nao_identificado').tolist()
        acertos[sid] = [1 if yt == yp else 0 for yt, yp in zip(y_true, y_pred)]

    # Friedman test — testa se pelo menos um sistema é diferente
    grupos = [acertos[sid] for sid in colunas_sistemas]
    if len(grupos) >= 3:
        stat_f, p_f = friedmanchisquare(*grupos)
        print(f'Friedman test:')
        print(f'  Estatística : {stat_f:.4f}')
        print(f'  p-valor     : {p_f:.4f}')
        print(f'  Conclusão   : {"Diferença significativa entre sistemas (p<0.05)" if p_f < 0.05 else "Sem diferença significativa (p>=0.05)"}')
    else:
        print('Friedman requer >= 3 sistemas.')

    # Wilcoxon post-hoc — compara pares
    print(f'\nWilcoxon post-hoc (com correção Bonferroni):')
    pares = list(combinations(colunas_sistemas, 2))
    alpha_corrigido = 0.05 / len(pares)
    print(f'  Alpha corrigido (Bonferroni): {alpha_corrigido:.4f}\n')

    resultados_wilcoxon = []
    for s1, s2 in pares:
        a1 = acertos[s1]
        a2 = acertos[s2]
        try:
            if a1 == a2:
                p_w = 1.0
            else:
                _, p_w = wilcoxon(a1, a2, zero_method='wilcox')
        except Exception:
            p_w = float('nan')

        sig = '✔ sig.' if p_w < alpha_corrigido else '—'
        media1 = round(sum(a1)/len(a1), 4)
        media2 = round(sum(a2)/len(a2), 4)
        melhor = s1 if media1 > media2 else s2

        resultados_wilcoxon.append({
            'Par': f'{s1} vs {s2}',
            'Acc S1': media1,
            'Acc S2': media2,
            'Melhor': melhor,
            'p-valor': round(p_w, 4),
            'Significativo': sig,
        })

    df_wilcoxon = pd.DataFrame(resultados_wilcoxon)
    df_wilcoxon.to_csv(os.path.join(OUTPUT_DIR, 'wilcoxon_posthoc.csv'), index=False, encoding='utf-8-sig')
    display(df_wilcoxon)
else:
    print('Execute as células 6 e 7 primeiro.')

## Célula 9 — Visualizações

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
CORES = {'basic': '#2E86AB', 'local': '#3BB273', 'global': '#E84855', 'drift': '#F4A261'}

if 'df_metricas' in dir() and len(df_metricas) > 0:

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Comparação dos 8 Sistemas — GraphRAG Grid', fontsize=14, fontweight='bold')

    sistemas_labels = df_metricas['Sistema'].tolist()
    cores_barras = [CORES[m] for m in df_metricas['Modo'].tolist()]

    # F1 macro com IC bootstrap
    ax = axes[0, 0]
    bars = ax.bar(sistemas_labels, df_metricas['F1 macro'], color=cores_barras, edgecolor='white')
    yerr_low  = (df_metricas['F1 macro'] - df_metricas['F1 IC 2.5%']).tolist()
    yerr_high = (df_metricas['F1 IC 97.5%'] - df_metricas['F1 macro']).tolist()
    ax.errorbar(sistemas_labels, df_metricas['F1 macro'],
                yerr=[yerr_low, yerr_high], fmt='none', color='black', capsize=4)
    ax.set_title('F1 Macro com IC 95% (bootstrap)')
    ax.set_ylabel('F1 macro')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

    # Accuracy
    ax = axes[0, 1]
    ax.bar(sistemas_labels, df_metricas['Accuracy'], color=cores_barras, edgecolor='white')
    ax.set_title('Accuracy por Sistema')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

    # Cohen's Kappa
    ax = axes[1, 0]
    ax.bar(sistemas_labels, df_metricas["Cohen's Kappa"], color=cores_barras, edgecolor='white')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax.set_title("Cohen's Kappa por Sistema")
    ax.set_ylabel('Kappa')
    ax.grid(axis='y', alpha=0.3)

    # MCC
    ax = axes[1, 1]
    ax.bar(sistemas_labels, df_metricas['MCC'], color=cores_barras, edgecolor='white')
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax.set_title('Matthews Correlation Coefficient (MCC)')
    ax.set_ylabel('MCC')
    ax.grid(axis='y', alpha=0.3)

    # Legenda de modos
    patches = [mpatches.Patch(color=c, label=m) for m, c in CORES.items()]
    fig.legend(handles=patches, loc='lower center', ncol=4, title='Modo de busca',
               bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig_grid_metricas.png'), bbox_inches='tight')
    plt.show()

    # Heatmap de acertos por prontuário × sistema
    if 'df_pivot' in dir():
        df_eval = df_pivot[df_pivot['doenca_principal'] != 'nenhuma'].copy()
        y_true  = df_eval['doenca_principal'].tolist()
        heatmap_data = pd.DataFrame(index=df_eval['prontuario_id'])
        for sid in colunas_sistemas:
            y_pred = df_eval[sid].fillna('nao_identificado').tolist()
            heatmap_data[sid] = [1 if yt == yp else 0 for yt, yp in zip(y_true, y_pred)]

        fig, ax = plt.subplots(figsize=(12, max(6, len(heatmap_data)//8)))
        sns.heatmap(heatmap_data.T, cmap='RdYlGn', vmin=0, vmax=1,
                    cbar_kws={'label': '1=acerto / 0=erro'},
                    xticklabels=False, ax=ax)
        ax.set_title('Acertos por Prontuário × Sistema', fontsize=13, fontweight='bold')
        ax.set_ylabel('Sistema')
        ax.set_xlabel('Prontuários (141)')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'fig_heatmap_acertos.png'), bbox_inches='tight')
        plt.show()
else:
    print('Execute as células 6 e 7 primeiro.')

## Célula 10 — Concordância entre modos (Inter-Mode Agreement)

In [ ]:
if 'df_pivot' in dir() and len(colunas_sistemas) > 1:
    df_eval = df_pivot[df_pivot['doenca_principal'] != 'nenhuma'].copy()

    def ima(row, cols):
        """Inter-Mode Agreement: proporção de modos que concordam na doença mais votada."""
        votos = [row[c] for c in cols if pd.notna(row[c]) and row[c] != '']
        if not votos:
            return 0
        moda = max(set(votos), key=votos.count)
        return votos.count(moda) / len(votos)

    df_eval['ima'] = df_eval.apply(lambda r: ima(r, colunas_sistemas), axis=1)
    df_eval['acerto_algum'] = df_eval.apply(
        lambda r: any(r[s] == r['doenca_principal'] for s in colunas_sistemas), axis=1
    )
    df_eval['acerto_todos'] = df_eval.apply(
        lambda r: all(r[s] == r['doenca_principal'] for s in colunas_sistemas), axis=1
    )

    print('Inter-Mode Agreement (IMA):')
    print(f'  IMA médio          : {df_eval["ima"].mean():.4f}')
    print(f'  IMA mediano        : {df_eval["ima"].median():.4f}')
    print(f'  Todos concordam    : {df_eval["ima"].eq(1.0).sum()} prontuários ({df_eval["ima"].eq(1.0).mean()*100:.1f}%)')
    print(f'  Ao menos 1 acerta  : {df_eval["acerto_algum"].sum()} prontuários ({df_eval["acerto_algum"].mean()*100:.1f}%)')
    print(f'  Todos acertam      : {df_eval["acerto_todos"].sum()} prontuários ({df_eval["acerto_todos"].mean()*100:.1f}%)')
    print(f'  Todos erram        : {(~df_eval["acerto_algum"]).sum()} prontuários ({(~df_eval["acerto_algum"]).mean()*100:.1f}%)')

    # Correlação IMA × acerto
    from scipy.stats import pointbiserialr
    corr, p_corr = pointbiserialr(df_eval['ima'], df_eval['acerto_algum'].astype(int))
    print(f'\n  Correlação IMA × acerto: r={corr:.4f}, p={p_corr:.4f}')
    print(f'  (alta concordância entre modos prediz acerto?)')

    df_eval[['prontuario_id', 'doenca_principal', 'ima', 'acerto_algum', 'acerto_todos'] + colunas_sistemas].to_csv(
        os.path.join(OUTPUT_DIR, 'analise_concordancia.csv'), index=False, encoding='utf-8-sig'
    )
    print(f'\n✔ Análise de concordância salva.')
else:
    print('Execute as células anteriores primeiro.')